# Dashboard Data Chat

This notebook powers the first dashboard chatbot. It uses retrieval-style context built from the current local price data, market ranking, and strategy backtests. This is preferable to fine-tuning a general model on a small, frequently changing dataset: each response sees the latest dashboard facts.

## Configuration

The dashboard reads `.env` through `python-dotenv`. Set `LLM_PROVIDER=gemini`, `anthropic`, or `openai`; if omitted, it selects the first configured provider.

In [ ]:
from dashboard.backend.main import (
    _active_llm_provider,
    _format_dashboard_chat_context,
    _llm_chat,
    feed,
)

symbol = feed.symbols[0]
print(f'Provider: {_active_llm_provider()}')
print(f'Selected symbol: {symbol}')

In [ ]:
context = _format_dashboard_chat_context(symbol, include_strategies=True)
print(context)

In [ ]:
system_prompt = (
    'You are a helpful Egyptian equities dashboard assistant. Use only the supplied '
    'dashboard facts, cite the relevant figures, and say when data is unavailable. '
    'This is educational information, not financial advice.'
)
question = 'Which strategy had the best return and what risk did it take?'
reply = _llm_chat(
    [{'role': 'user', 'content': f'{context}\n\nUSER QUESTION\n{question}'}],
    system=system_prompt,
    symbol=symbol,
)
print(reply)

## Why this is not fine-tuning

Fine-tuning teaches a model response style using many labelled examples; it does not safely keep prices, rankings, or backtests current. The dashboard instead rebuilds factual context for every request, while the provider’s model handles the natural-language reasoning.